# AI Chef Agent — LoRA Fine-tuning (Colab One-click Runner)

**Runtime requirement**: Google Colab (free T4 GPU is enough)

**Files to download after completion**:
- `data/lora_adapter/` — LoRA adapter weights
- `data/lora_eval_report.json` — evaluation metrics
- `data/lora_dataset/` — generated training dataset (backup)

---
**Run order**: Step 0 → Step 1 → Step 2 → Step 3 → Step 4 (download)

## Step 0: Environment Setup

In [ ]:
# check GPU availability
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU model: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# install dependencies
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 datasets>=2.18.0 accelerate>=0.28.0 bitsandbytes>=0.43.0 openai>=1.30.0 tqdm

In [ ]:
import os

# upload project scripts (click 'Choose Files' after running this cell)
# upload these three files:
# - dataset_builder.py
# - train_lora.py
# - evaluate.py
from google.colab import files
files.upload()

In [ ]:
# set up API key (needed for Self-Instruct data expansion & LLM-as-Judge evaluation)
import os
from google.colab import userdata

# option 1 (recommended): add DASHSCOPE_API_KEY in Colab's left-side 'Secrets' panel
try:
    os.environ["DASHSCOPE_API_KEY"] = userdata.get("DASHSCOPE_API_KEY")
    print("✅ API Key loaded from Secrets")
except Exception:
    # option 2: paste directly here (don't commit this to Git)
    os.environ["DASHSCOPE_API_KEY"] = "your_dashscope_api_key_here"
    print("⚠️  Please replace with your real DASHSCOPE_API_KEY")

# create data directories
os.makedirs("data/lora_dataset", exist_ok=True)
os.makedirs("data/lora_adapter", exist_ok=True)
print("✅ Directory structure created")

## Step 1: Build Dataset

- Loads 100 built-in seed samples
- Calls Qwen API to expand via Self-Instruct to ~1000 samples
- Splits into train / val / test and saves as JSONL

Estimated time: **5~10 minutes** (depends on API response speed)

In [ ]:
!python dataset_builder.py

In [ ]:
# verify dataset
import json
for split in ["train", "val", "test"]:
    path = f"data/lora_dataset/{split}.jsonl"
    with open(path) as f:
        count = sum(1 for _ in f)
    print(f"{split:5s}: {count} samples")

## Step 2: QLoRA Fine-tuning

- Base model: Qwen2.5-7B-Instruct
- QLoRA: 4-bit NF4 quantization + LoRA (r=8, alpha=16, target q/k/v/o_proj)
- Training epochs: 3

Estimated time: **60~90 minutes** (Colab T4)

In [ ]:
!python train_lora.py

In [ ]:
import os
import json

# verify adapter was saved
adapter_files = os.listdir("data/lora_adapter")
print("✅ Saved adapter files:", adapter_files)

# read training summary
with open("data/lora_adapter/train_summary.json") as f:
    summary = json.load(f)

elapsed = summary['elapsed_seconds']
minutes, seconds = divmod(int(elapsed), 60)

print("\n📊 Training Results:")
print(f"⏱️ Time elapsed: {minutes}m {seconds}s ({elapsed}s)")
print(f"📉 Final loss: {summary['final_train_loss']}")

## Step 3: Evaluation — Base vs Fine-tuned

Compares base Qwen2.5-7B vs QLoRA fine-tuned model on 100 test samples.

Estimated time: **15~20 minutes**

In [ ]:
!python evaluate.py

In [ ]:
# print key metrics
with open("data/lora_eval_report.json") as f:
    report = json.load(f)

base  = report.get("base_model") or {}
ft    = report["finetuned_model"]
judge = report.get("judge_model", "qwen-max-latest")

def fmt(val):
    return f"{val:.1f}%" if val is not None else "N/A"

def delta(b, f):
    if b is None or f is None: return "N/A"
    d = f - b
    sign = "↑" if d >= 0 else "↓"
    return f"{sign}{abs(d):.1f} pp"

b_chef = base.get("chef_accuracy_pct")
f_chef = ft.get("chef_accuracy_pct")
b_safe = base.get("food_safety_accuracy_pct")
f_safe = ft.get("food_safety_accuracy_pct")

print("=" * 58)
print("       QLoRA Fine-tuning Results — Quick Summary")
print("=" * 58)
print(f"Judge model     : {judge}")
print(f"Test samples    : {ft['sample_count']}")
print()
print(f"Chef suggestion accuracy : {fmt(b_chef)} → {fmt(f_chef)}   ({delta(b_chef, f_chef)})")
print(f"Food safety accuracy     : {fmt(b_safe)} → {fmt(f_safe)}   ({delta(b_safe, f_safe)})")
print()
print("[Resume summary]")
if f_chef is not None and b_chef is not None and f_safe is not None and b_safe is not None:
    d1 = f_chef - b_chef
    d2 = f_safe - b_safe
    print(f"Built a 1000-sample chef instruction dataset via Self-Instruct,")
    print(f"fine-tuned Qwen2.5-7B-Instruct with QLoRA (4-bit NF4, r=8);")
    if d1 >= 0:
        print(f"chef suggestion accuracy improved from {b_chef:.0f}% to {f_chef:.0f}% (+{d1:.1f} pp) using {judge} as judge;")
    else:
        print(f"chef suggestion accuracy: {b_chef:.0f}% → {f_chef:.0f}%;")
    if d2 >= 0:
        print(f"food safety accuracy improved from {b_safe:.0f}% to {f_safe:.0f}% (+{d2:.1f} pp).")
    else:
        print(f"food safety accuracy: {b_safe:.0f}% → {f_safe:.0f}%.")

## Step 4: Package & Download

Zip up all outputs and download to your local machine. Unzip and place the `data/` folder into the project directory.

In [ ]:
import shutil
from google.colab import files

# zip everything up
shutil.make_archive("lora_outputs", "zip", ".", "data")
print("✅ Packaged as lora_outputs.zip")

# download
files.download("lora_outputs.zip")
print("✅ Download started — unzip and merge the data/ folder into your local project")